<a href="https://colab.research.google.com/github/Sowmya-Sirisanagandla/Customer-Churn-Prediction-Retention-Analysis./blob/main/Customer_Churn_Prediction_%26_Retention_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load data
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Drop junk column
df = df.drop(columns=['customerID'])

# Fix numeric column
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop missing values
df = df.dropna()

# Sanity checks
df.isnull().sum()
df.info()
(df['MonthlyCharges'] == 0).sum()
df[df['tenure'] == 0][['tenure', 'TotalCharges']].head()

# NOW split features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

#encoding for target col churn...
y = y.map({'No': 0, 'Yes': 1})
y.value_counts()
X = pd.get_dummies(X, drop_first=True)
X.info()



<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   object 
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   object 
 3   Dependents        7032 non-null   object 
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   object 
 6   MultipleLines     7032 non-null   object 
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    7032 non-null   object 
 9   OnlineBackup      7032 non-null   object 
 10  DeviceProtection  7032 non-null   object 
 11  TechSupport       7032 non-null   object 
 12  StreamingTV       7032 non-null   object 
 13  StreamingMovies   7032 non-null   object 
 14  Contract          7032 non-null   object 
 15  PaperlessBilling  7032 non-null   object 
 16  PaymentMethod     7032 non-null   object 
 17  

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)


In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


X_train: (4922, 30)
y_train: (4922,)
X_val: (1055, 30)
y_val: (1055,)
X_test: (1055, 30)
y_test: (1055,)


In [ ]:
# CLASS DISTRIBUTION
print("Train churn ratio:")
print(y_train.value_counts(normalize=True))

print("\nValidation churn ratio:")
print(y_val.value_counts(normalize=True))

print("\nTest churn ratio:")
print(y_test.value_counts(normalize=True))


Train churn ratio:
Churn
0    0.734254
1    0.265746
Name: proportion, dtype: float64

Validation churn ratio:
Churn
0    0.733649
1    0.266351
Name: proportion, dtype: float64

Test churn ratio:
Churn
0    0.734597
1    0.265403
Name: proportion, dtype: float64


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# 1. Initialize model (handle class imbalance)
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

# 2. Train on training data
model.fit(X_train, y_train)

# 3. Predict on validation data
y_val_pred = model.predict(X_val)
y_val_prob = model.predict_proba(X_val)[:, 1]

# 4. Evaluation
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_val_prob))


Confusion Matrix:
 [[561 213]
 [ 54 227]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.72      0.81       774
           1       0.52      0.81      0.63       281

    accuracy                           0.75      1055
   macro avg       0.71      0.77      0.72      1055
weighted avg       0.81      0.75      0.76      1055

ROC-AUC: 0.8532994013627963


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
#Extract top churn drivers using coefficients.
import pandas as pd

coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_[0]
})

print("Top 10 features by positive coefficient:")
print(coef_df.sort_values("coefficient", ascending=False).head(10))

print("\nTop 10 features by negative coefficient:")
print(coef_df.sort_values("coefficient").head(10))

Top 10 features by positive coefficient:
                                  feature  coefficient
8          MultipleLines_No phone service     0.524013
10            InternetService_Fiber optic     0.407605
28         PaymentMethod_Electronic check     0.383356
26                   PaperlessBilling_Yes     0.291014
9                       MultipleLines_Yes     0.211748
0                           SeniorCitizen     0.167871
23                    StreamingMovies_Yes     0.079076
29             PaymentMethod_Mailed check     0.078834
21                        StreamingTV_Yes     0.075441
27  PaymentMethod_Credit card (automatic)     0.051830

Top 10 features by negative coefficient:
                                 feature  coefficient
25                     Contract_Two year    -1.360687
24                     Contract_One year    -0.713728
19                       TechSupport_Yes    -0.501083
13                    OnlineSecurity_Yes    -0.498870
7                       PhoneService_Yes  

In [ ]:
#Threshold tuning
import numpy as np
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_val, y_val_prob)

# Show few threshold options
for t, p, r in zip(thresholds[::50], precision[::50], recall[::50]):
    print(f"Threshold={t:.2f} | Precision={p:.2f} | Recall={r:.2f}")


Threshold=0.00 | Precision=0.27 | Recall=1.00
Threshold=0.01 | Precision=0.28 | Recall=1.00
Threshold=0.03 | Precision=0.29 | Recall=1.00
Threshold=0.05 | Precision=0.31 | Recall=0.99
Threshold=0.07 | Precision=0.32 | Recall=0.99
Threshold=0.10 | Precision=0.34 | Recall=0.98
Threshold=0.15 | Precision=0.36 | Recall=0.98
Threshold=0.19 | Precision=0.38 | Recall=0.96
Threshold=0.25 | Precision=0.40 | Recall=0.94
Threshold=0.33 | Precision=0.43 | Recall=0.92
Threshold=0.38 | Precision=0.45 | Recall=0.89
Threshold=0.43 | Precision=0.48 | Recall=0.86
Threshold=0.49 | Precision=0.51 | Recall=0.83
Threshold=0.56 | Precision=0.54 | Recall=0.78
Threshold=0.62 | Precision=0.58 | Recall=0.73
Threshold=0.67 | Precision=0.63 | Recall=0.68
Threshold=0.72 | Precision=0.68 | Recall=0.61
Threshold=0.77 | Precision=0.72 | Recall=0.52
Threshold=0.81 | Precision=0.73 | Recall=0.40
Threshold=0.84 | Precision=0.74 | Recall=0.27
Threshold=0.87 | Precision=0.74 | Recall=0.14
Threshold=0.92 | Precision=1.00 | 

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

chosen_threshold = 0.43

y_val_pred_custom = (y_val_prob >= chosen_threshold).astype(int)

print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred_custom))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_custom))


Confusion Matrix:
 [[507 267]
 [ 39 242]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.66      0.77       774
           1       0.48      0.86      0.61       281

    accuracy                           0.71      1055
   macro avg       0.70      0.76      0.69      1055
weighted avg       0.81      0.71      0.73      1055



In [ ]:
y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= 0.43).astype(int)

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_test_prob))


Confusion Matrix:
 [[508 267]
 [ 51 229]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.66      0.76       775
           1       0.46      0.82      0.59       280

    accuracy                           0.70      1055
   macro avg       0.69      0.74      0.68      1055
weighted avg       0.79      0.70      0.72      1055

ROC-AUC: 0.8224170506912443


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Train
rf.fit(X_train, y_train)

# Validation predictions
y_val_prob_rf = rf.predict_proba(X_val)[:, 1]
y_val_pred_rf = (y_val_prob_rf >= 0.43).astype(int)  # SAME threshold logic

# Evaluation
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred_rf))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_rf))
print("ROC-AUC:", roc_auc_score(y_val, y_val_prob_rf))


Confusion Matrix:
 [[538 236]
 [ 42 239]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.70      0.79       774
           1       0.50      0.85      0.63       281

    accuracy                           0.74      1055
   macro avg       0.72      0.77      0.71      1055
weighted avg       0.81      0.74      0.75      1055

ROC-AUC: 0.8476164859720268
